# SignBridge PSL — Pakistan Sign Language Letters Model

This notebook downloads a PSL alphabet dataset, extracts MediaPipe hand + pose landmarks
(159-dim: both hands + upper body), trains an MLP classifier, and exports it for the browser.

**Expected runtime:** ~30–60 minutes on Colab CPU
**Output:** `psl-default/` folder (model.json + seed.json) saved to Google Drive

---
### What this notebook does
1. Installs dependencies & clones the SignBridge repo
2. Downloads PSL alphabet dataset from Kaggle (wasifullahcs/dataset-pakistani-sign-language)
3. Extracts MediaPipe landmarks (159-dim: both hands + upper-body pose)
4. Trains a 2-layer MLP classifier (~300K params)
5. Exports to signbridge-mlp-weights-v1 format (TF.js compatible)
6. Saves everything to your Google Drive

### Prerequisites
- A Kaggle API key (kaggle.json) uploaded to Colab
  - Go to https://www.kaggle.com/settings/account → Create New API Token
  - Upload kaggle.json to /content/ in Colab

In [ ]:
# ── Cell 1: Install dependencies ───────────────────────────────────────
!pip install -q mediapipe==0.10.35 opencv-python>=4.10.0 scikit-learn numpy Pillow
!pip install -q kaggle

import os, json, sys, shutil
from collections import Counter
from pathlib import Path

print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/signbridge-psl-letters'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Artifacts will be saved to: {DRIVE_OUTPUT}")

In [ ]:
# ── Cell 3: Clone SignBridge repo ──────────────────────────────────────
REPO_URL = 'https://github.com/mhmdtaha091/SignBridge.git'
REPO_DIR = '/content/SignBridge'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}/ml
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Cell 4: Upload Kaggle API key ──────────────────────────────────────
# Upload your kaggle.json file to /content/ first.
# If you haven't already, go to Kaggle → Settings → API → Create New Token.

!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/ 2>/dev/null || echo "⚠ kaggle.json not found. Upload it from your computer."
!chmod 600 ~/.kaggle/kaggle.json 2>/dev/null

# Test the API key
try:
    !kaggle datasets list --sort-by=updated 2>/dev/null | head -3
    print("✅ Kaggle API key works!")
except:
    print("⚠ Kaggle API key not set up. Upload kaggle.json to /content/ and re-run.")

In [ ]:
# ── Cell 5: Download PSL alphabet dataset from Kaggle ──────────────────
DATASET = 'wasifullahcs/dataset-pakistani-sign-language'
DATA_DIR = '/content/psl_dataset'

if not os.path.exists(DATA_DIR):
    print(f"Downloading {DATASET}...")
    !kaggle datasets download -d {DATASET} -p /content/psl_zip --unzip
    # The dataset extracts into folders per class
    # Rename to our expected directory
    import glob
    extracted = glob.glob('/content/psl_zip/*')
    if extracted:
        shutil.move('/content/psl_zip', DATA_DIR)
        print(f"✅ Dataset extracted to {DATA_DIR}")
        # List what we got
        for d in sorted(os.listdir(DATA_DIR))[:10]:
            items = len(os.listdir(os.path.join(DATA_DIR, d))) if os.path.isdir(os.path.join(DATA_DIR, d)) else 0
            print(f"  {d}: {items} files")
    else:
        print("⚠ Unexpected extraction structure. Listing:")
        !ls /content/psl_zip/
else:
    print(f"✅ Dataset already at {DATA_DIR}")
    # Show existing folders
    folders = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"  {len(folders)} folders found:")
    for f in sorted(folders)[:10]:
        items = len(os.listdir(os.path.join(DATA_DIR, f)))
        print(f"    {f}: {items} images")

In [ ]:
# ── Cell 6: Map PSL dataset folder names to A-Z labels ─────────────────
# The Kaggle dataset has Urdu/Persian alphabet folder names.
# We create a mapping from folder names to English letters.
# If folder names don't map cleanly, we use an index-based fallback.

import re

def build_label_map(dataset_dir):
    """Build a mapping from folder names to A-Z labels."""
    folders = sorted([
        d for d in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, d))
    ])

    print(f"Found {len(folders)} folders")

    label_map = {}
    # Manual mapping for common Urdu/Persian folder names to English letters
    # This is a best-effort mapping — adjust based on actual folder names
    english_letters = [chr(c) for c in range(ord('A'), ord('Z') + 1)]

    # Try to extract letters from folder names
    for folder in folders:
        clean = folder.strip().upper()
        # Try single-letter match
        if len(clean) == 1 and clean in english_letters:
            label_map[folder] = clean
        # Try extracting first ASCII letter
        else:
            match = re.search(r'[A-Z]', clean)
            if match:
                label_map[folder] = match.group()

    # For unmatched folders, assign sequentially from remaining letters
    unmatched = [f for f in folders if f not in label_map]
    used = set(label_map.values())
    available = [l for l in english_letters if l not in used]

    for i, folder in enumerate(unmatched):
        if i < len(available):
            label_map[folder] = available[i]
            print(f"  Auto-mapped: '{folder}' → '{available[i]}'")
        else:
            # Cycle through letters
            label_map[folder] = english_letters[i % 26]
            print(f"  Cycled: '{folder}' → '{english_letters[i % 26]}'")

    print(f"\nLabel map ({len(label_map)} entries):")
    for folder, letter in sorted(label_map.items(), key=lambda x: x[1]):
        print(f"  {folder:30s} → {letter}")

    return label_map

LABEL_MAP = build_label_map(DATA_DIR)

# Save the mapping for reference
with open('/content/label_map.json', 'w') as f:
    json.dump(LABEL_MAP, f, indent=2)
print("✅ Label map saved to /content/label_map.json")

In [ ]:
# ── Cell 7: Extract MediaPipe landmarks ─────────────────────────────────
# Runs extract_psl_landmarks.py with the downloaded dataset.

import subprocess

# First, create a symlink so the script finds the dataset
os.symlink(DATA_DIR, '/content/psl_images', target_is_directory=True)

EXTRACT_CMD = [
    sys.executable, 'extract_psl_landmarks.py',
    '/content/psl_images',
]

print("Running:", ' '.join(EXTRACT_CMD))
print()
result = subprocess.run(EXTRACT_CMD)
if result.returncode != 0:
    print(f"\n❌ Extraction failed (exit code {result.returncode})")
    sys.exit(1)
print("\n✅ Landmark extraction complete")

In [ ]:
# ── Cell 8: Inspect extracted landmarks ────────────────────────────────
OUT_PATH = os.path.join('data', 'psl_landmarks.jsonl')

if not os.path.exists(OUT_PATH):
    print(f"❌ {OUT_PATH} not found. Extraction may have failed.")
else:
    labels = []
    with open(OUT_PATH) as f:
        for line in f:
            rec = json.loads(line)
            labels.append(rec['label'])

    counts = Counter(labels)
    print(f"Total samples: {len(labels):,}")
    print(f"Unique letters: {len(counts)}")
    print(f"Feature dimension: 159 (two hands + pose)")
    print()
    print("Per-letter distribution:")
    for letter, count in counts.most_common():
        bar = '█' * min(40, count // max(1, counts.most_common(1)[0][1] // 40))
        print(f"  {letter}: {count:5d}  {bar}")

    if len(counts) < 5:
        print(f"\n⚠ Only {len(counts)} letters — model quality will be limited.")
        print("  Consider adding more training data or recording your own.")

In [ ]:
# ── Cell 9: Train the PSL letters MLP ──────────────────────────────────
# Architecture: Input(159) → Dense(128) → Dropout(0.3) → Dense(64) → Dropout(0.3) → Softmax(26)
# Same architecture as ASL, but with 159-dim input for two-handed PSL signs.

TRAIN_CMD = [
    sys.executable, 'train_psl_letters.py',
]

print("Running:", ' '.join(TRAIN_CMD))
print()
result = subprocess.run(TRAIN_CMD)
if result.returncode != 0:
    print(f"\n❌ Training failed (exit code {result.returncode})")
    sys.exit(1)
print("\n✅ Training complete")

In [ ]:
# ── Cell 10: Inspect trained model ─────────────────────────────────────
MODEL_PATH = os.path.join('..', 'web', 'public', 'models', 'psl-default', 'model.json')
SEED_PATH = os.path.join('..', 'web', 'public', 'seed', 'psl-fingerspelling.json')

with open(MODEL_PATH) as f:
    model = json.load(f)

print(f"Model format:    {model['format']}")
print(f"Feature size:    {model['featureSize']}")
print(f"Labels ({len(model['labels'])}):       {''.join(model['labels'])}")
print(f"Val accuracy:    {model.get('valAccuracy', 'N/A')}")
print(f"Architecture:    Input({model['featureSize']}) → Dense(128) → Dropout → Dense(64) → Softmax")
print(f"Model file:      {MODEL_PATH} ({os.path.getsize(MODEL_PATH) // 1024} KB)")
print(f"Seed file:       {SEED_PATH}")
if os.path.exists(SEED_PATH):
    with open(SEED_PATH) as f:
        seed = json.load(f)
    print(f"Seed samples:    {len(seed)}")

In [ ]:
# ── Cell 11: Save to Google Drive ──────────────────────────────────────
import shutil
from datetime import datetime

DEST = os.path.join(DRIVE_OUTPUT, 'psl-default')
if os.path.exists(DEST):
    shutil.rmtree(DEST)

shutil.copytree(os.path.dirname(MODEL_PATH), DEST)

# Also copy the seed
SEED_DEST = os.path.join(DRIVE_OUTPUT, 'psl-fingerspelling.json')
if os.path.exists(SEED_PATH):
    shutil.copy(SEED_PATH, SEED_DEST)

# Write a summary
summary_path = os.path.join(DRIVE_OUTPUT, 'psl_training_summary.txt')
with open(summary_path, 'w') as f:
    f.write("SignBridge — PSL Letters Model\n")
    f.write(f"Trained: {datetime.now().strftime('%Y-%m-%d %H:%M UTC')}\n")
    f.write(f"Dataset: Pakistan Sign Language Alphabet (Kaggle)\n")
    f.write(f"Architecture: MLP(159→128→Dropout→64→Dropout→Softmax)\n")
    f.write(f"Input: 159-dim (both hands 63+63 + pose upper 33)\n")
    f.write(f"Classes: {model['labels']}\n")
    f.write(f"Validation accuracy: {model.get('valAccuracy', 'N/A')}\n")

print(f"✅ Saved to Google Drive: {DRIVE_OUTPUT}")
print()
print("Files:")
for root, dirs, files in os.walk(DRIVE_OUTPUT):
    for fname in files:
        path = os.path.join(root, fname)
        print(f"  {os.path.relpath(path, DRIVE_OUTPUT)}  ({os.path.getsize(path):,} bytes)")

---
## Next Steps

### 1. Download the model from Google Drive
The files are at `MyDrive/signbridge-psl-letters/` in your Google Drive.
Download:
- `psl-default/model.json` ← place in `SignBridge/web/public/models/psl-default/`
- `psl-fingerspelling.json` ← place in `SignBridge/web/public/seed/`

### 2. Verify in the browser
```bash
cd SignBridge/web
npm run dev
```
Open http://localhost:5173, switch to PSL 🇵🇰, go to Learn → the PSL alphabet shows.
Go to Interpret (ABC mode) to test fingerspelling recognition.

### 3. Record more data for missing letters
Use the SignBridge Data Studio in PSL mode to record your own hand samples.
The more data you record, the better the model — especially for motion letters (J, Z).

### 4. Train the PSL word-sign model
Run `ml/colab/signbridge_psl_words.ipynb` to train the GRU word-sign model
on the Dynamic Word-Level Pakistan Sign Language dataset.

In [ ]:
# ── Cell 12: Print label order (for vocab.ts verification) ─────────────
print("Model output label order (index → label):")
print()
for i, label in enumerate(model['labels']):
    print(f"  {i:2d}: '{label}'")

print()
print("Make sure your pslVocab.ts PSL_LETTERS array has these same letters.")
print("The model labels and vocab labels MUST match (order doesn't matter).")